# Runner Detection and Tracking with YOLOv8 and ByteTrack

This notebook demonstrates an end-to-end computer vision pipeline for detecting and tracking runners in a race scene.

**Pipeline:** dataset loading → YOLOv8 training → validation → multi-object tracking with ByteTrack.

The dataset contains a single class: `runner/person`.

> Note: API keys, local file paths, private identifiers, and the source video are intentionally excluded from this public notebook.


## 1. Install Dependencies

This notebook is designed to run in Google Colab or another Python environment with GPU support.


In [ ]:
!pip install -q ultralytics roboflow


## 2. Check GPU


In [ ]:
!nvidia-smi


## 3. Import Libraries and Verify Ultralytics


In [ ]:
from ultralytics import YOLO, checks

checks()


## 4. Load the Dataset from Roboflow

For security, the Roboflow API key is not stored in this notebook.

In Google Colab, set your key as an environment variable before running this cell:

```python
import os
os.environ["ROBOFLOW_API_KEY"] = "YOUR_PRIVATE_KEY"
```

Do **not** commit your real API key to GitHub.


In [ ]:
import os
from roboflow import Roboflow

api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    raise ValueError(
        "ROBOFLOW_API_KEY is not set. Add it as an environment variable before running this cell."
    )

rf = Roboflow(api_key=api_key)

# Update these values if your Roboflow workspace/project/version changes.
project = rf.workspace("ali-cnz0i").project("alii")
version = project.version(2)

dataset = version.download("yolov8")
print("Dataset location:", dataset.location)


## 5. Train YOLOv8

Training configuration used in the original experiment:

- Model: `YOLOv8s`
- Epochs: `80`
- Patience: `25`
- Batch size: `16`
- Image size: `768`
- Cosine learning-rate schedule enabled


In [ ]:
!yolo task=detect mode=train \
  model=yolov8s.pt \
  data={dataset.location}/data.yaml \
  epochs=80 \
  patience=25 \
  batch=16 \
  imgsz=768 \
  cos_lr=True \
  save=True \
  project=/content/runs/detect \
  name=train


## 6. Inspect Training Results


In [ ]:
import os
import glob
from IPython.display import Image, display

train_dir = "/content/runs/detect/train"

print("Training directory contents:")
print(os.listdir(train_dir))

confusion_matrix = os.path.join(train_dir, "confusion_matrix.png")
results_plot = os.path.join(train_dir, "results.png")

if os.path.exists(confusion_matrix):
    display(Image(filename=confusion_matrix, width=600))

if os.path.exists(results_plot):
    display(Image(filename=results_plot, width=600))

for img in glob.glob(os.path.join(train_dir, "*.jpg"))[:5]:
    display(Image(filename=img, height=600))


## 7. Validate the Trained Model


In [ ]:
!yolo task=detect mode=val \
  model=/content/runs/detect/train/weights/best.pt \
  data={dataset.location}/data.yaml


## 8. Multi-Object Tracking with ByteTrack

The trained detector is combined with ByteTrack to maintain object identities across consecutive video frames.

The source movie/video is **not included** in this repository. Upload your own test video to the runtime and set `VIDEO_IN` accordingly.


In [ ]:
import os
import glob
from ultralytics import YOLO
from IPython.display import Video, display

VIDEO_IN = "/content/data_test.mp4"
MODEL_PATH = "/content/runs/detect/train/weights/best.pt"

IMG_SIZE = 768
CONF_TH = 0.63
IOU_TH = 0.5
AGNOSTIC_NMS = False
TRACKER = "bytetrack.yaml"

OUT_MP4 = "/content/data_test_track.mp4"

if not os.path.exists(VIDEO_IN):
    raise FileNotFoundError(
        f"Video not found at {VIDEO_IN}. Upload a test video and update VIDEO_IN."
    )

model = YOLO(MODEL_PATH)

print("Loaded model:", MODEL_PATH)
print("Class names:", model.names)

results = model.track(
    source=VIDEO_IN,
    imgsz=IMG_SIZE,
    conf=CONF_TH,
    iou=IOU_TH,
    agnostic_nms=AGNOSTIC_NMS,
    tracker=TRACKER,
    persist=True,
    save=True
)

save_dir = results[0].save_dir
print("Tracking results saved to:", save_dir)

video_files = (
    glob.glob(os.path.join(save_dir, "*.mp4"))
    + glob.glob(os.path.join(save_dir, "*.avi"))
)

if not video_files:
    raise RuntimeError(f"No output video found in {save_dir}")

tracked_video = video_files[0]

if tracked_video.endswith(".avi"):
    os.system(
        f'ffmpeg -y -i "{tracked_video}" -vcodec libx264 -acodec aac "{OUT_MP4}"'
    )
else:
    os.system(f'cp "{tracked_video}" "{OUT_MP4}"')

print("Final tracked video:", OUT_MP4)
display(Video(OUT_MP4, width=700))


## 9. Optional: Download the Result in Google Colab


In [ ]:
try:
    from google.colab import files
    files.download(OUT_MP4)
except ImportError:
    print("This cell is intended for Google Colab. The output is saved at:", OUT_MP4)


## Notes

- Only the model-development code is published here.
- The original movie/video is not distributed with this repository.
- Keep credentials such as Roboflow API keys outside source control.
- Large datasets, trained weights, and generated videos should generally not be committed directly to GitHub.
